# PubChem loader statistics

This notebook explores the local `PubChemAssayLoader` catalog and draws a few dataset-level and graph-level statistics. It is intended as a quick inspection workflow before choosing an assay for downstream modeling.


## Data prerequisite

This notebook inspects local assay exports. If the selected assay is missing, first run [`prepare_pubchem_assay.ipynb`](../data_preparation/prepare_pubchem_assay.ipynb); it writes the active/inactive SDF files to `data-local/PUBCHEM/`. Parsed graph caches are stored under `data-local/graph_corpus_cache/`.


In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from abstractgraph_graphicalizer.chem import PubChemAssayLoader


## Catalog overview

`PubChemAssayLoader.list_assays()` already exposes file sizes and molecule counts for each paired active/inactive assay export. We convert those summaries into a dataframe first.


In [ ]:
pubchem_root = repo_root / "data-local" / "PUBCHEM"
loader = PubChemAssayLoader(pubchem_root, on_error="skip")
summaries = loader.list_assays()

if not summaries:
    raise FileNotFoundError(f"No paired PubChem assay exports found under {loader.root}")

assay_table = pd.DataFrame(
    {
        "assay_id": summary.assay_id,
        "active_molecule_count": summary.active_molecule_count,
        "inactive_molecule_count": summary.inactive_molecule_count,
        "total_molecule_count": summary.total_molecule_count,
        "active_size_mb": summary.active_size_bytes / 1024 / 1024,
        "inactive_size_mb": summary.inactive_size_bytes / 1024 / 1024,
        "total_size_mb": summary.total_size_bytes / 1024 / 1024,
        "active_fraction": summary.active_molecule_count / summary.total_molecule_count,
        "inactive_fraction": summary.inactive_molecule_count / summary.total_molecule_count,
    }
    for summary in summaries
).sort_values(["total_molecule_count", "assay_id"], ascending=[False, True]).reset_index(drop=True)

print("root:", loader.root)
display(assay_table.head(10))
assay_table.describe(include="all").transpose()


In [ ]:
top_n = min(12, len(assay_table))
plot_table = assay_table.head(top_n).sort_values("total_molecule_count")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].barh(plot_table["assay_id"], plot_table["active_molecule_count"], label="active", color="#1f77b4")
axes[0].barh(
    plot_table["assay_id"],
    plot_table["inactive_molecule_count"],
    left=plot_table["active_molecule_count"],
    label="inactive",
    color="#ff7f0e",
)
axes[0].set_title(f"Top {top_n} assays by molecule count")
axes[0].set_xlabel("molecules")
axes[0].set_ylabel("assay_id")
axes[0].legend()

axes[1].scatter(
    assay_table["inactive_molecule_count"],
    assay_table["active_molecule_count"],
    s=np.clip(assay_table["total_size_mb"] * 8, 40, 300),
    alpha=0.7,
    color="#2ca02c",
)
limits = [0, max(assay_table["inactive_molecule_count"].max(), assay_table["active_molecule_count"].max())]
axes[1].plot(limits, limits, linestyle="--", color="black", linewidth=1)
axes[1].set_title("Class balance across assays")
axes[1].set_xlabel("inactive molecules")
axes[1].set_ylabel("active molecules")

for _, row in assay_table.head(min(8, len(assay_table))).iterrows():
    axes[1].annotate(row["assay_id"], (row["inactive_molecule_count"], row["active_molecule_count"]), fontsize=8)

plt.tight_layout()


## Load one assay and inspect graph statistics

By default the notebook uses the largest assay currently available. You can override `assay_id` or the per-class limits below if you want a smaller slice.


In [ ]:
assay_id = assay_table.loc[0, "assay_id"]
limit_active = 500
limit_inactive = 500

active_graphs, inactive_graphs = loader.load_split(
    assay_id,
    limit_active=limit_active,
    limit_inactive=limit_inactive,
)

print("assay_id:", assay_id)
print("active graphs  :", len(active_graphs))
print("inactive graphs:", len(inactive_graphs))


In [ ]:
graphs = active_graphs + inactive_graphs
labels = ["active"] * len(active_graphs) + ["inactive"] * len(inactive_graphs)

graph_stats = pd.DataFrame(
    {
        "activity": labels,
        "node_count": [graph.number_of_nodes() for graph in graphs],
        "edge_count": [graph.number_of_edges() for graph in graphs],
    }
)
graph_stats["average_degree"] = 2.0 * graph_stats["edge_count"] / graph_stats["node_count"].clip(lower=1)
graph_stats["density"] = 2.0 * graph_stats["edge_count"] / (
    graph_stats["node_count"] * (graph_stats["node_count"] - 1)
).where(graph_stats["node_count"] > 1, 1)

display(graph_stats.head())
graph_stats.groupby("activity").agg(["mean", "median", "min", "max"])


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for label, color in (("active", "#1f77b4"), ("inactive", "#ff7f0e")):
    subset = graph_stats.loc[graph_stats["activity"] == label]
    axes[0, 0].hist(subset["node_count"], bins=20, alpha=0.6, label=label, color=color)
    axes[0, 1].hist(subset["edge_count"], bins=20, alpha=0.6, label=label, color=color)
    axes[1, 0].hist(subset["average_degree"], bins=20, alpha=0.6, label=label, color=color)
    axes[1, 1].scatter(subset["node_count"], subset["edge_count"], alpha=0.45, s=20, label=label, color=color)

axes[0, 0].set_title("Node count distribution")
axes[0, 0].set_xlabel("nodes")
axes[0, 0].set_ylabel("graphs")

axes[0, 1].set_title("Edge count distribution")
axes[0, 1].set_xlabel("edges")
axes[0, 1].set_ylabel("graphs")

axes[1, 0].set_title("Average degree distribution")
axes[1, 0].set_xlabel("average degree")
axes[1, 0].set_ylabel("graphs")

axes[1, 1].set_title("Edges vs nodes")
axes[1, 1].set_xlabel("nodes")
axes[1, 1].set_ylabel("edges")

for axis in axes.ravel()[:3]:
    axis.legend()
axes[1, 1].legend()

plt.tight_layout()


In [ ]:
summary = (
    graph_stats.groupby("activity")
    .agg(
        graphs=("activity", "size"),
        mean_nodes=("node_count", "mean"),
        median_nodes=("node_count", "median"),
        mean_edges=("edge_count", "mean"),
        median_edges=("edge_count", "median"),
        mean_density=("density", "mean"),
    )
    .round(3)
)

summary
